# AI 5102 - Exercise 2: Prompt Engineering and In-Context Learning
### Plaksha University · Introduction to Large Language Models and Generative AI · Exercise 2

In this assignment you will design and **systematically compare** prompting strategies —
**zero-shot**, **few-shot**, **chain-of-thought (CoT)** — and apply **self-consistency** to
improve reliability, on a small reasoning/classification task. Rather than testing one strategy
on one model, you will run every strategy across a **ladder of models** ranging from a small
open model to a frontier reasoning model.

**Why the model ladder matters.** A common claim is "chain-of-thought always helps." That claim
is roughly true for *small, non-reasoning* models — but it is often **false** for frontier models,
and especially false for dedicated *reasoning models* (e.g. OpenAI's o-series, DeepSeek-R1), which
already perform extensive internal reasoning before they even start writing their visible answer.
If you only test CoT on a GPT-4-class model, you might conclude "CoT doesn't matter." If you only
test it on a 1B model, you might conclude "CoT always matters." **Both conclusions are wrong** — the
truth depends on where the model sits on the capability ladder. This assignment is built so you can
see that pattern directly in your own data, not just read about it.

## Learning objectives
By the end of this assignment you will be able to:
- Implement zero-shot, few-shot, and chain-of-thought prompt builders for a reasoning/classification task
- Implement **self-consistency** (sample-many-times, majority-vote) and explain the reliability/cost trade-off
- Build a small **evaluation harness** that measures the accuracy of a `(model, strategy)` pair on a dataset
- Run a **model x strategy** grid experiment and read a results table/plot
- Explain *where* CoT and self-consistency help, *where* they stop helping, and *why* — tying the
  answer to how reasoning models work internally

## Outcomes
This assignment maps to course learning outcomes **CLO2** (prompt engineering / in-context learning)
and **CLO3** (systematic empirical evaluation of LLM behavior).

## The classification harness
The classification harness below (`test_reviews`, prompt construction, label extraction) pairs with an **API-based model ladder**, so nearly
the same evaluation logic runs unchanged against everything from a 1B model to a frontier reasoning
model — you only ever change the `model=` string.

## How this notebook is organized
- **Part 0 — Setup.** Gateway + API key.
- **Part 1 — Dataset.** A small labeled reasoning task (word problems requiring 2 steps of arithmetic/logic).
- **Part 2 — Prompt strategies.** `zero_shot`, `few_shot`, `chain_of_thought` prompt builders + answer extraction.
- **Part 3 — Self-consistency.** Sample k times, take the majority vote.
- **Part 4 — Evaluation harness.** `evaluate(...)` + the model x strategy grid experiment.
- **Part 5 — Analysis.** You interpret the grid and explain the pattern.
- **Part 6 — Exercises + rubric.**

Cells marked **`GIVEN CODE`** are complete and runnable as-is (the exercises build on them). Cells marked **`STUDENT TODO`** are where you write code.


---
## Part 0 — Setup

We use the **OpenAI Python SDK**, pointed at an **NVIDIA's OpenAI-compatible gateway** instead of OpenAI's
own servers. Because these gateways speak the same request/response shape as the OpenAI API,
switching models is *just a string* — no new SDK, no new request format, no new parsing code.

NVIDIA NIM API (https://integrate.api.nvidia.com/v1) — provides access to a range of open and reasoning-focused models from families such as Llama, Qwen, and GPT-OSS through a single API endpoint. This is what we use below.

Get an NVIDIA API key from the NVIDIA Developer portal. NVIDIA provides a free API tier for experimenting with its hosted models, subject to usage and rate limits.

The model IDs in MODELS are examples and may change over time. NVIDIA regularly updates its available model catalog, so if a model ID is unavailable, replace it with a currently supported equivalent from NVIDIA's API catalog. What matters for this assignment is that the four slots span small → larger/frontier → reasoning-oriented models, not the exact model names.


In [ ]:
# Install the OpenAI SDK -- the same SDK works for ANY OpenAI-compatible gateway
%pip install -q openai


In [ ]:
from getpass import getpass
from openai import OpenAI

print("Enter your NVIDIA API key:")
NVIDIA_API_KEY = getpass()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    timeout=60,      # fail fast instead of hanging for the SDK's 10-min default
    max_retries=2,   # transient errors get retried automatically
)

# --- The model ladder: small -> frontier -> reasoning ------------------------
# PLACEHOLDERS: if an ID 404s, check https://build.nvidia.com/models and swap in a
# current equivalent. The rest of the notebook only cares about the *string*.
MODELS = [
    "meta/llama-3.1-8b-instruct",    # small(ish), non-reasoning
    "meta/llama-3.1-70b-instruct",   # large, non-reasoning (SLOW on the free tier: ~25s+/call)
    "openai/gpt-oss-20b",            # REASONING model
]
# NOTE: verified live on integrate.api.nvidia.com (Aug 2026). Optional 4th rung:
# "deepseek-ai/deepseek-v4-flash-0731" works but is slow/flaky on the free tier
# (~20s+ per call) -- add it back if you have time. Smaller rungs such as
# meta/llama-3.2-1b-instruct exist in the catalog but time out on the free
# serverless tier; swap one in if it starts responding, the contrast is starker.

# Quick connectivity check
r = client.chat.completions.create(
    model=MODELS[0],
    messages=[
        {"role": "user", "content": "Say 'ready' and nothing else."}
    ],
    max_tokens=5,
)

print(r.choices[0].message.content)


**A note on cost, latency, and rate limits.** The full model x strategy grid in Part 4 makes on
the order of a few hundred small API calls. At current OpenRouter prices for these models that's
well under one US dollar total, but:
- Debug with `MODELS[:2]` and a 4–5 example slice of the dataset first.
- If you hit a rate limit, add a short `time.sleep(...)` inside the evaluation loop.
- Reasoning models (like `o4-mini`) are slower per call — expect that column of the grid to take
  noticeably longer than the others.


---
## Part 1 — Dataset: multi-step arithmetic word problems

Chain-of-thought and self-consistency show their biggest effect on tasks that need **more than one
inferential step** — a model can't just pattern-match the surface form, it has to actually compose
intermediate results. Plain sentiment classification (single-step, and near-ceiling for most models)
is a poor task for showing a CoT effect, so here we use short **word problems that require two
arithmetic/logic steps** to reach a final numeric answer. (You reuse the *sentiment* task from the harness below in Exercise 6.3 as a contrast case — a task that's easy enough that CoT should do little.)

Each example has:
- `"question"`: a short word problem
- `"answer"`: the gold final answer, as a **string** of the integer (so string-equality is exact)

This is intentionally small (20 items) so the grid experiment in Part 4 stays fast and cheap.
Feel free to extend it for Exercise 6.1.


In [ ]:
# ============================================================
# GIVEN CODE -- dataset of 20 multi-step word problems
# ============================================================

reasoning_dataset = [
    {"question": "A warehouse has 8 crates of 45 water bottles each and 6 crates of 30 bottles each. A store buys one-third of all the bottles, then returns 15 of them. How many bottles does the warehouse have now?", "answer": "375"},
    {"question": "Lena buys 140 bracelets at $3 each and sells 85 of them at $7 each. She also pays a $50 stall fee. What is her profit so far in dollars?", "answer": "125"},
    {"question": "A bus route is 84 km one way. The bus makes the round trip 3 times a day, 6 days a week. Fuel costs $0.25 per km. What is the weekly fuel cost in dollars?", "answer": "756"},
    {"question": "Three friends split a restaurant bill of $96 equally, and each also adds a $4 tip. Later, one friend gets a $12 refund. How much did that friend pay in the end?", "answer": "24"},
    {"question": "A tank holds 600 liters and is 75% full. It leaks 12 liters per day for 5 days, then 90 liters are added. How many liters are in the tank now?", "answer": "480"},
    {"question": "A school orders 15 boxes of 24 pencils each. Each of its 18 classrooms receives 16 pencils. How many pencils remain?", "answer": "72"},
    {"question": "Amir saves $35 per week for 8 weeks, then spends $95 on a jacket, then saves $20 per week for 6 more weeks. How much does he have now?", "answer": "305"},
    {"question": "A printer prints 42 pages per minute. A 3,150-page job pauses for 5 minutes halfway through for a paper refill. How many minutes does the whole job take?", "answer": "80"},
    {"question": "A cinema has 12 rows of 18 seats. Adult tickets cost $9 and child tickets cost $5. A show sells out, with 150 adult tickets and the rest children's. What is the total revenue in dollars?", "answer": "1680"},
    {"question": "A farmer plants 9 rows of 32 seedlings. 25% of the seedlings fail. Each surviving plant yields 4 kg of tomatoes. The farmer keeps 100 kg and sells the rest at $2 per kg. How many dollars does the sale earn?", "answer": "1528"},
    {"question": "Nina bikes at 14 km/h for 90 minutes, rests, then bikes at 10 km/h for 30 minutes. How many kilometers does she cover in total?", "answer": "26"},
    {"question": "A factory makes 240 gadgets per day and 5% are defective. The non-defective gadgets are packed 12 to a box. How many FULL boxes are packed per day?", "answer": "19"},
    {"question": "A library had 1,450 books. It removed 6 boxes of 40 damaged books each, and bought 3 sets of 75 new books each. How many books does it have now?", "answer": "1435"},
    {"question": "Sam is paid $18 per hour, plus time-and-a-half for every hour beyond 40 in a week. He worked 46 hours this week. What is his pay in dollars?", "answer": "882"},
    {"question": "A recipe for 6 people needs 450 grams of flour. You are cooking for 10 people and already have 300 grams. How many more grams must you buy?", "answer": "450"},
    {"question": "A phone battery is at 20%. Charging adds 15 percentage points every 10 minutes. After 40 minutes of charging, the phone is used for 30 minutes, draining 1 percentage point every 2 minutes. What is the battery percentage now?", "answer": "65"},
    {"question": "A car park charges $4 for the first hour and $2.50 for each additional hour. A driver parks from 09:00 to 15:00. What is the fee in dollars?", "answer": "16.5"},
    {"question": "Four painters each paint 3 walls per hour. A job has 132 walls. After 8 hours, two painters leave. How many MORE hours do the remaining painters need to finish?", "answer": "6"},
    {"question": "Tickets numbered 1 to 500 are sold in order. Every 5th buyer gets a keychain and every 8th buyer gets a poster. How many buyers get BOTH a keychain and a poster?", "answer": "12"},
    {"question": "A store discounts a $250 jacket by 20%, then a 10% sales tax is added at checkout. What is the final price in dollars?", "answer": "220"},
]

# A separate, small pool of worked examples for few-shot / CoT demonstrations.
# (Kept disjoint from reasoning_dataset so we never evaluate on an example the
# model has already seen in-context.)
reasoning_fewshot_pool = [
    {
        "question": "A vendor buys 10 crates of oranges at $8 each and sells all the oranges for a total of $150. What is the vendor's profit?",
        "answer": "70",
        "cot": "The vendor spends 10 * $8 = $80 on crates. Selling all the oranges brings in $150. Profit is revenue minus cost: $150 - $80 = $70.",
    },
    {
        "question": "A pool holds 2,000 liters. A pump fills it at 250 liters per hour. How many hours until the pool is 3/4 full?",
        "answer": "6",
        "cot": "3/4 full means 0.75 * 2000 = 1500 liters. At 250 liters/hour, time = 1500 / 250 = 6 hours.",
    },
    {
        "question": "A school bus has 4 rows of 6 seats. If 9 seats are empty, how many students are on the bus?",
        "answer": "15",
        "cot": "Total seats = 4 * 6 = 24. Occupied seats = total minus empty = 24 - 9 = 15.",
    },
]

print(f"Reasoning dataset: {len(reasoning_dataset)} problems")
print(f"Few-shot/CoT demonstration pool: {len(reasoning_fewshot_pool)} problems")
print()
print("Example item:", reasoning_dataset[0])


### The sentiment task, reused as a contrast case
The `test_reviews` set — 20 movie reviews, binary
positive/negative. It's a **single-step** classification task most models already do well on
zero-shot, which is exactly why it's a useful contrast: in Exercise 6.3 you'll check whether CoT
helps here as much as it helps on the multi-step arithmetic task above.


In [ ]:
# ============================================================
# 20 movie reviews, binary positive/negative.
# ============================================================
test_reviews = [
    # Clear positive
    {"text": "This movie was absolutely fantastic! The acting was superb.", "label": "positive"},
    {"text": "A masterpiece of modern cinema. I was moved to tears.", "label": "positive"},
    {"text": "The cinematography was breathtaking and the story compelling.", "label": "positive"},
    {"text": "An instant classic! The director outdid themselves.", "label": "positive"},
    {"text": "I laughed, I cried, and I want to watch it again.", "label": "positive"},
    # Subtler positive
    {"text": "Not what I expected, but in the best possible way.", "label": "positive"},
    {"text": "A rare film that is both thoughtful and well-crafted.", "label": "positive"},
    {"text": "Despite a slow start, the payoff was absolutely worth it.", "label": "positive"},
    {"text": "The kind of movie that stays with you for days afterward.", "label": "positive"},
    {"text": "Finally, a sequel that lives up to the original.", "label": "positive"},
    # Clear negative
    {"text": "Terrible waste of time. The script was lazy and predictable.", "label": "negative"},
    {"text": "Boring, predictable, and poorly acted from start to finish.", "label": "negative"},
    {"text": "Save your money. Worst film this year by far.", "label": "negative"},
    {"text": "Confusing plot, bad pacing, and the ending made no sense.", "label": "negative"},
    {"text": "Two hours I'll never get back. Avoid at all costs.", "label": "negative"},
    # Subtler negative
    {"text": "It had potential but squandered it with poor execution.", "label": "negative"},
    {"text": "The trailer was better than the actual movie.", "label": "negative"},
    {"text": "Great cast completely wasted on a mediocre script.", "label": "negative"},
    {"text": "Started strong but completely fell apart in the second half.", "label": "negative"},
    {"text": "I really wanted to like this more than I did.", "label": "negative"},
]

# Few-shot examples for the sentiment task (disjoint from test_reviews)
few_shot_examples = [
    {"text": "I loved every minute of it. Highly recommended!", "label": "positive"},
    {"text": "What a disappointment. I expected so much more.", "label": "negative"},
    {"text": "Beautiful storytelling and amazing performances.", "label": "positive"},
    {"text": "Dull and uninspired from beginning to end.", "label": "negative"},
]

print(f"Sentiment test set: {len(test_reviews)} reviews")
print(f"Sentiment few-shot pool: {len(few_shot_examples)} reviews")


---
## Part 2 — Prompt strategy builders

Each strategy below is a function `strategy(item, examples=None) -> str` that returns the
**user-message text** to send to the model for one dataset item. Keeping a single call signature
across strategies is what lets the Part 4 harness loop over `(model, strategy)` pairs generically.

- `zero_shot(item)` — task instructions only, no worked examples.
- `few_shot(item, examples)` — task instructions + a handful of solved `(question, answer)` pairs.
- `chain_of_thought(item, examples)` — like few-shot, but demonstrations show **reasoning steps**
  before the final answer, and the instructions ask the model to "think step by step." The model's
  own response is expected to end with a line of the form `Answer: <value>` so we can extract it
  reliably even when the response also contains reasoning text.

All three end the prompt by asking for a specific final-line format, because **how you parse the
output is part of prompt design** — a strategy that reasons well but can't be parsed loses to a
strategy that reasons less but is parsed perfectly.


In [ ]:
# ============================================================
# GIVEN CODE -- prompt strategy builders for the reasoning task
# ============================================================

def zero_shot(item, examples=None):
    # Zero-shot prompt: instructions + the question only. `examples` is ignored;
    # kept in the signature so every strategy has the same call shape.
    return (
        "Solve the following math word problem. "
        "Respond with ONLY the final numeric answer on the last line, "
        "in the exact form 'Answer: <number>'.\n\n"
        f"Question: {item['question']}\n"
    )


def few_shot(item, examples=None):
    # Few-shot prompt: instructions + worked (question, answer) pairs (no reasoning shown)
    # + the target question. `examples` is a list of dicts with 'question' and 'answer' keys,
    # e.g. a slice of `reasoning_fewshot_pool`.
    prompt = (
        "Solve math word problems. "
        "Respond with ONLY the final numeric answer on the last line, "
        "in the exact form 'Answer: <number>'.\n\n"
    )
    if examples:
        for ex in examples:
            prompt += f"Question: {ex['question']}\nAnswer: {ex['answer']}\n\n"
    prompt += f"Question: {item['question']}\n"
    return prompt


def chain_of_thought(item, examples=None):
    # Chain-of-thought prompt: instructions ask the model to reason step by step;
    # demonstrations (if given) SHOW reasoning before their answer line. `examples` is a
    # list of dicts with 'question', 'cot' (worked reasoning), and 'answer' keys, e.g.
    # `reasoning_fewshot_pool`. With `examples=None` this is "zero-shot CoT" (just the
    # 'think step by step' instruction, no worked demonstrations).
    prompt = (
        "Solve math word problems. Think step by step, showing your reasoning, "
        "then give the final numeric answer on the LAST line in the exact form "
        "'Answer: <number>'.\n\n"
    )
    if examples:
        for ex in examples:
            prompt += (
                f"Question: {ex['question']}\n"
                f"Reasoning: {ex['cot']}\n"
                f"Answer: {ex['answer']}\n\n"
            )
    prompt += f"Question: {item['question']}\nReasoning:"
    return prompt


# Quick sanity check -- print one prompt from each strategy
print("=" * 60, "\nZERO-SHOT PROMPT\n", "=" * 60, sep="")
print(zero_shot(reasoning_dataset[0]))
print("=" * 60, "\nFEW-SHOT PROMPT (2 examples)\n", "=" * 60, sep="")
print(few_shot(reasoning_dataset[0], reasoning_fewshot_pool[:2]))
print("=" * 60, "\nCHAIN-OF-THOUGHT PROMPT (1 example)\n", "=" * 60, sep="")
print(chain_of_thought(reasoning_dataset[0], reasoning_fewshot_pool[:1]))


### Calling the model and extracting an answer

`generate(...)` wraps a single chat-completion call so every strategy talks to the gateway the
same way. `extract_answer(...)` pulls a final numeric answer out of raw model text: it looks for
`"Answer: <value>"` first (which handles chain-of-thought responses that show reasoning before the
answer), and falls back to the **last number in the text** if that exact phrase is missing (which
handles zero-shot/few-shot responses that sometimes drop the label).


In [ ]:
# ============================================================
# GIVEN CODE -- model call + answer extraction
# ============================================================
import re

import time
from openai import RateLimitError, APIError

def generate(model, prompt, temperature=0.0, max_tokens=300):
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content

def extract_answer(text):
    # Extract a final numeric answer from raw model text.
    #
    # Priority 1: a line/phrase of the form 'Answer: <number>' (handles CoT output
    # that shows reasoning before the final line).
    # Priority 2 (fallback): the LAST number anywhere in the text (handles terse
    # zero-shot/few-shot responses that omit the 'Answer:' label).
    # Returns the answer as a string, or 'unknown' if no number is found at all.
    if text is None:
        return "unknown"

    # Priority 1: explicit "Answer: <value>" (case-insensitive, allows $ , commas)
    m = re.search(r"answer\s*[:=]\s*\$?(-?[\d,]+(?:\.\d+)?)", text, re.IGNORECASE)
    if m:
        return m.group(1).replace(",", "").rstrip(".")

    # Priority 2: fall back to the last number that appears anywhere
    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)
    if nums:
        return nums[-1].replace(",", "").rstrip(".")

    return "unknown"


def normalize_numeric(s):
    # Normalize a numeric-looking string for comparison ('7.0' == '7' == '7').
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except (ValueError, TypeError):
        return s


# Sanity check the extractor on a few handwritten examples (no API calls)
_examples = [
    ("Let's break it down. 10 * 8 = 80. 150 - 80 = 70.\nAnswer: 70", "70"),
    ("Answer: $1,250", "1250"),
    ("The total comes to 42.", "42"),
    ("I'm not sure.", "unknown"),
]
for text, expected in _examples:
    got = extract_answer(text)
    status = "OK" if normalize_numeric(got) == normalize_numeric(expected) else "MISMATCH"
    print(f"[{status}] extract_answer({text!r}) = {got!r} (expected {expected!r})")


---
## Part 3 — Self-consistency

**Self-consistency** (Wang et al., 2022, https://arxiv.org/abs/2203.11171) samples the SAME prompt
`k` times at a non-zero temperature, extracts an answer from each sample, and returns the **majority
vote**. The intuition: a model's reasoning paths are noisy, but wrong paths tend to disagree with each
other while correct paths tend to agree, so voting cancels out a lot of that noise. It costs `k` times
the compute/latency of a single call — a real trade-off you're asked to quantify in Exercise 6.2.

Self-consistency is a **wrapper**: it takes any strategy's prompt-generation, calls the model `k`
times, and reduces the `k` extracted answers to one. This is exactly why every strategy shares
`(item, examples=None) -> prompt` as its call signature — `self_consistency` can wrap any of them.


In [ ]:
# ============================================================
# GIVEN CODE -- self-consistency via sampling + majority vote
# ============================================================
from collections import Counter


def majority_vote(answers):
    # Return the most common item in `answers`. Ties are broken by the item that
    # appears EARLIEST in the list (i.e. earliest sample wins), so the result is a
    # deterministic function of the input order -- important for testability.
    if not answers:
        return "unknown"
    counts = Counter(answers)
    best_count = max(counts.values())
    for a in answers:  # preserves first-seen order among the tied winners
        if counts[a] == best_count:
            return a
    return "unknown"  # unreachable, but keeps the function total


def self_consistency(model, prompt_fn, item, examples=None, k=5, temperature=0.7,
                      generate_fn=None, extract_fn=None):
    # Sample `prompt_fn(item, examples)` from `model` k times at `temperature`,
    # extract an answer from each sample with `extract_fn`, and return the majority
    # vote plus the raw per-sample answers (useful for inspecting disagreement).
    #
    # `generate_fn` / `extract_fn` default to the real `generate` / `extract_answer`
    # but can be overridden -- this is what lets the unit tests exercise the voting
    # logic with a fake, deterministic "model" and no network access.
    #
    # Returns: (winning_answer: str, all_answers: list[str])
    generate_fn = generate_fn or generate
    extract_fn = extract_fn or extract_answer

    prompt = prompt_fn(item, examples)
    samples = [generate_fn(model, prompt, temperature=temperature) for _ in range(k)]
    answers = [extract_fn(s) for s in samples]
    return majority_vote(answers), answers


# Sanity check with a FAKE generate_fn (no API calls) -- see also the real unit
# tests in tests/test_a2.py, which exercise this exact logic offline.
_fake_responses = iter(["Answer: 70", "Answer: 68", "Answer: 70", "Answer: 70", "Answer: 12"])
_fake_generate = lambda model, prompt, temperature=0.7: next(_fake_responses)
winner, all_answers = self_consistency(
    "fake-model", chain_of_thought, reasoning_fewshot_pool[0], k=5,
    generate_fn=_fake_generate,
)
print("Per-sample answers:", all_answers)
print("Majority vote      :", winner, "(expected 70)")


---
## Part 4 — Evaluation harness and the model x strategy grid

`evaluate(model, strategy_fn, dataset, ...)` runs one strategy against one model over an entire
dataset and returns accuracy (fraction correct). It is deliberately generic: `strategy_fn` can be
`zero_shot`, `few_shot`, `chain_of_thought`, OR a `self_consistency`-wrapped version of any of
them — `evaluate` doesn't need to know which.


In [ ]:
# ============================================================
# GIVEN CODE -- generic evaluation harness
# ============================================================

def evaluate(model, strategy_fn, dataset, examples=None, generate_fn=None,
             extract_fn=None, temperature=0.0, verbose=False):
    # Run `strategy_fn` against every item in `dataset` using `model`, and return
    # (accuracy: float in [0, 1], predictions: list[str]).
    #
    # strategy_fn(item, examples) -> prompt string   e.g. zero_shot, few_shot, chain_of_thought
    # generate_fn/extract_fn default to the real API call + regex extractor, but can be
    # swapped for fakes in tests. `dataset` items must have a 'question' (or 'text') key
    # and an 'answer' (or 'label') key -- ground truth.
    generate_fn = generate_fn or generate
    extract_fn = extract_fn or extract_answer

    predictions = []
    correct = 0
    for item in dataset:
        prompt = strategy_fn(item, examples)
        raw = generate_fn(model, prompt, temperature=temperature)
        pred = extract_fn(raw)
        predictions.append(pred)

        gold = item.get("answer", item.get("label"))
        is_correct = normalize_numeric(pred) == normalize_numeric(gold)
        correct += int(is_correct)

        if verbose:
            mark = "OK" if is_correct else "WRONG"
            q = item.get("question", item.get("text", ""))[:50]
            print(f"[{mark:5}] {q}... | pred={pred!r} gold={gold!r}")

    return correct / len(dataset), predictions


# Quick smoke test on ONE model, ONE strategy, a 3-item slice -- cheap, confirms
# the plumbing works end to end before committing to the full grid below.
acc, preds = evaluate(MODELS[0], zero_shot, reasoning_dataset[:3])
print(f"Smoke test -- {MODELS[0]} / zero-shot / 3 items: {acc:.0%}")
print("Predictions:", preds)


In [ ]:
# ============================================================
# GIVEN CODE -- the model x strategy grid experiment
# ============================================================
import time
import pandas as pd

FEWSHOT_K = 3          # number of worked examples for few-shot / CoT
SELF_CONSISTENCY_K = 5  # number of samples for self-consistency

# Each strategy: (label, callable(model, dataset) -> accuracy)
def run_zero_shot(model, dataset):
    acc, _ = evaluate(model, zero_shot, dataset)
    return acc

def run_few_shot(model, dataset):
    acc, _ = evaluate(model, few_shot, dataset, examples=reasoning_fewshot_pool[:FEWSHOT_K])
    return acc

def run_cot(model, dataset):
    acc, _ = evaluate(model, chain_of_thought, dataset, examples=reasoning_fewshot_pool[:FEWSHOT_K])
    return acc

def run_cot_self_consistency(model, dataset):
    correct = 0
    for item in dataset:
        winner, _ = self_consistency(
            model, chain_of_thought, item, examples=reasoning_fewshot_pool[:FEWSHOT_K],
            k=SELF_CONSISTENCY_K,
        )
        correct += int(normalize_numeric(winner) == normalize_numeric(item["answer"]))
    return correct / len(dataset)

STRATEGIES = {
    "zero-shot": run_zero_shot,
    "few-shot": run_few_shot,
    "chain-of-thought": run_cot,
    "CoT + self-consistency": run_cot_self_consistency,
}

# NOTE: this is the expensive cell. While developing, shrink MODELS and/or use
# reasoning_dataset[:5] first, then widen once your code is verified to work.

results = []

# Use only 5 questions while testing NVIDIA API limits.
# HEADS UP: even at 5 questions, the full grid can take 20-45 minutes
# depending on API load (measured ~40 min on a free-tier key). The 70B model
# is by far the slowest (~25s+ per call, and its CoT calls sometimes time out
# entirely) -- a printed "Skipped ... | ..." line is EXPECTED behavior, not a
# bug: the try/except below records the failure and moves on.
# Start it, then work on the written exercises while it runs (or finish
# it as take-home).
reasoning_dataset_small = reasoning_dataset[:5]

for model in MODELS:
    for strategy_name, strategy_run in STRATEGIES.items():
        t0 = time.time()

        try:
            acc = strategy_run(model, reasoning_dataset_small)
        except Exception as e:
            print(f"Skipped {model} | {strategy_name}: {e}")
            continue

        elapsed = time.time() - t0

        results.append({
            "model": model,
            "strategy": strategy_name,
            "accuracy": acc,
            "seconds": elapsed
        })

        print(
            f"{model:35s} | {strategy_name:24s} | "
            f"acc={acc:.0%} | {elapsed:.1f}s"
        )

        time.sleep(10)

results_df = pd.DataFrame(results)
results_df


### Visualizing the grid

A grouped bar chart with **models on the x-axis** and **one bar per strategy** makes the pattern
easy to see: strategy effects should shrink (bars converge) as you move from the small model to the
reasoning model.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pivot = results_df.pivot(index="model", columns="strategy", values="accuracy").loc[MODELS]

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(pivot.index))
width = 0.8 / len(pivot.columns)
colors = ['#ff6b6b', '#feca57', '#48dbfb', '#1dd1a1']

for i, strategy_name in enumerate(pivot.columns):
    ax.bar(x + i * width, pivot[strategy_name] * 100, width, label=strategy_name, color=colors[i % len(colors)])

ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
ax.set_xticklabels(pivot.index, rotation=15, ha="right")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 105)
ax.set_title("Prompting strategy x model: where does CoT / self-consistency actually help?")
ax.legend(title="Strategy")
plt.tight_layout()
plt.show()

print(pivot.round(2))


---
## Part 5 — Analysis (written, required before the exercises)

Look at the `results_df` table and the grouped bar chart above, and answer the following in a
markdown cell. This analysis is graded on **reasoning quality**, not on getting a "correct" ranking —
your numbers will vary run to run because of sampling noise and whichever placeholder models you
ended up using.

1. **Where did chain-of-thought help the most, and where did it help the least (or not at all)?**
   Name the specific model(s) for each.
2. **Did self-consistency add anything on top of plain chain-of-thought?** For which model(s), if any?
   Was the extra accuracy (if there was any) worth 5x the API calls?
3. **Reasoning models.** For the reasoning model in your ladder (e.g. `o4-mini` or whichever
   reasoning model you substituted), how did zero-shot compare to chain-of-thought? If they're close,
   propose an explanation grounded in *how reasoning models work* (think about what happens between
   the prompt and the visible output — you don't need to have covered Lecture 9 yet to reason about
   this from what you've observed).
4. **If you had to deploy ONE (model, strategy) pair** for this task under a tight latency/cost
   budget, which would you pick, and what would you give up?

*Your answers:*

>


---
## Part 6 — Exercises (100 points total)

Exercises 6.1–6.3 are **code + written**; write your implementation in the indicated
`STUDENT TODO` cell, run it, and answer the accompanying questions in the markdown cell below it.


### Exercise 6.1 — Extend the dataset and re-run the grid (20 points) · **code + written**

The provided `reasoning_dataset` has 20 items. Small datasets are noisy: a single flipped
answer moves accuracy by 5 percentage points.

**Your task:**
a) (10 pts) Add **10 new** word problems to a new list `reasoning_dataset_extended` (so the original
   is untouched). Include at least 3 that need **three or more** inferential steps (harder than the
   instructor set) and at least 2 that are deliberately "trick" questions (e.g. irrelevant numbers,
   a step that looks necessary but isn't).
b) (5 pts) Re-run the Part 4 grid (`MODELS x STRATEGIES`) on `reasoning_dataset_extended`.
c) (5 pts) In 3-5 sentences: did the harder/trickier items change *which* strategies looked best, or
   just shift every number down uniformly? What does that tell you about the original dataset's
   difficulty?


In [ ]:
# === STUDENT TODO: Exercise 6.1(a) ===
reasoning_dataset_extended = reasoning_dataset + [
    # Add 10 new dicts here, each: {"question": "...", "answer": "..."}
    # At least 3 should need 3+ steps; at least 2 should be "trick" questions.
]

# TODO: after adding items, run this assertion (should pass once you've added 10 items)
# assert len(reasoning_dataset_extended) == len(reasoning_dataset) + 10


*Your written answer to 6.1(c):*

>


### Exercise 6.2 — Cost/accuracy trade-off curve for self-consistency (25 points) · **code + written**

Self-consistency's accuracy gain (if any) has a compute cost: `k` API calls instead of 1. This
exercise asks you to trace out that trade-off.

**Your task:**
a) (15 pts) Write `sweep_self_consistency(model, dataset, k_values)` that, for one `model` and one
   `dataset`, runs `chain_of_thought` self-consistency at every `k` in `k_values` (e.g. `[1, 3, 5, 9]`)
   and returns a dict `{k: accuracy}`. (`k=1` should reduce to plain chain-of-thought — no voting
   needed with a single sample.) Reuse `self_consistency` and `chain_of_thought` from Parts 2–3;
   do not reimplement the sampling loop from scratch.
b) (5 pts) Run your sweep for the **small** model and for the **reasoning** model in `MODELS`, and
   plot both curves (accuracy vs. k) on the same axes.
c) (5 pts) In 3-5 sentences: does accuracy keep climbing as `k` grows, or plateau? Does the
   plateau point differ between the small model and the reasoning model? What would you tell a
   teammate who wants to set `k=20` "to be safe"?


In [ ]:
# === STUDENT TODO: Exercise 6.2(a) ===
def sweep_self_consistency(model, dataset, k_values, examples=None, prompt_fn=None):
    # TODO: for each k in k_values, run chain_of_thought self-consistency (k=1 is
    # just chain_of_thought with no voting needed) across the WHOLE dataset and
    # record accuracy. Return {k: accuracy}.
    # Hint: reuse self_consistency(model, prompt_fn, item, examples=..., k=k) per item,
    # and normalize_numeric(...) to compare against item["answer"].
    prompt_fn = prompt_fn or chain_of_thought
    pass  # replace with your implementation


# === STUDENT TODO: Exercise 6.2(b) ===
# k_values = [1, 3, 5, 9]
# small_curve = sweep_self_consistency(MODELS[0], reasoning_dataset, k_values, examples=reasoning_fewshot_pool[:FEWSHOT_K])
# reasoning_curve = sweep_self_consistency(MODELS[-1], reasoning_dataset, k_values, examples=reasoning_fewshot_pool[:FEWSHOT_K])
#
# plt.figure(figsize=(8, 5))
# plt.plot(k_values, [small_curve[k] * 100 for k in k_values], marker="o", label=MODELS[0])
# plt.plot(k_values, [reasoning_curve[k] * 100 for k in k_values], marker="o", label=MODELS[-1])
# plt.xlabel("k (self-consistency samples)")
# plt.ylabel("Accuracy (%)")
# plt.title("Self-consistency: accuracy vs. sampling budget")
# plt.legend()
# plt.show()


*Your written answer to 6.2(c):*

>


### Exercise 6.3 — Does CoT help a task it wasn't designed for? (20 points) · **code + written**

Part 4's grid used the multi-step arithmetic task, which is exactly the kind of task CoT papers use to
show gains. Sentiment classification (`test_reviews`) is a **single-step** task most models
already handle well zero-shot. Does CoT still help there?

**Your task:**
a) (10 pts) Write a `sentiment_chain_of_thought(item, examples=None)` prompt builder, parallel to
   `chain_of_thought` above but for the sentiment task (ask the model to briefly reason about tone/
   word choice, then answer with `Answer: positive` or `Answer: negative` on the last line). Adapt
   `extract_answer` if needed, or write a small `extract_sentiment_answer` variant.
b) (5 pts) Run `evaluate` with plain zero-shot sentiment prompting AND your CoT version, for at least
   2 models from `MODELS` (one small, one frontier), on `test_reviews`.
c) (5 pts) In 3-5 sentences: was the CoT gain on sentiment smaller than the CoT gain you saw on the
   arithmetic task in Part 4? If so, connect that to *why* — what's different about a task that needs
   multi-step composition vs. one that's closer to pattern recognition?


In [ ]:
# === STUDENT TODO: Exercise 6.3(a) ===
def sentiment_chain_of_thought(item, examples=None):
    # TODO: build a CoT prompt for sentiment classification, parallel to chain_of_thought()
    # but for item['text'] / item['label'] instead of item['question'] / item['answer'].
    # End with an instruction to give the final answer as "Answer: positive" or "Answer: negative".
    pass  # replace with your implementation


def extract_sentiment_answer(text):
    # TODO: like extract_answer(), but look for "positive"/"negative" rather than a number.
    # Reuse the "Answer: <label>" pattern idea from extract_answer(); fall back to whichever
    # of "positive"/"negative" appears LAST in the text; return "unknown" if neither appears.
    pass  # replace with your implementation


# === STUDENT TODO: Exercise 6.3(b) ===
# for m in [MODELS[0], MODELS[2]]:
#     acc_zero, _ = evaluate(m, zero_shot, test_reviews, extract_fn=extract_sentiment_answer)
#     acc_cot, _ = evaluate(m, sentiment_chain_of_thought, test_reviews, extract_fn=extract_sentiment_answer)
#     print(f"{m}: zero-shot={acc_zero:.0%}  CoT={acc_cot:.0%}")


*Your written answer to 6.3(c):*

>


### Exercise 6.4 — Reflection (15 points) · **written**

1. **(5 pts)** Restate, in your own words, the "capability ladder" pattern this notebook was built
   to surface. Why is testing a single strategy on a single model an unreliable way to learn whether
   that strategy "works"?
2. **(5 pts)** Self-consistency assumes wrong reasoning paths disagree with each other more than
   correct ones do. Describe a task or failure mode where this assumption would break down (i.e.
   where a *majority* of samples could confidently agree on the *same wrong* answer).
3. **(5 pts)** If you were advising a team shipping an LLM feature with a real reasoning model
   (o-series/R1-class) in production, would you recommend they implement chain-of-thought prompting
   themselves? Would you recommend self-consistency? Justify both answers with a cost/benefit
   argument, not just "reasoning models don't need it."

*Your answers:*

>


### Exercise 6.5 — Error analysis: when does CoT win, and when does it lose? (bonus) · **code + written**

Accuracy numbers hide the interesting part: *which* questions flip between strategies. The
`evaluate` harness returns per-question predictions, so you can line the strategies up side by side.

**Your task:**
a) Run the starter cell below to build a per-question comparison table for the **small** model.
   Find **two questions where zero-shot and few-shot are both wrong but chain-of-thought is right**.
   For each, look at the wrong answers: they are usually a *legitimate intermediate value* from the
   problem (a partial computation), not a random number. Identify which step of the solution each
   wrong answer corresponds to.
b) Now hunt in the opposite direction, in both senses: are there any questions where **CoT is wrong
   but a simpler strategy (zero-/few-shot) on the same model is right**? And any where **the small
   model's CoT is wrong but a simpler/smaller configuration got it right** (compare against another
   model's column if you extend the starter)? Report what you find — *"none"* is a perfectly good
   empirical finding, but then explain **why** such inversions are rare, and describe at least one
   mechanism that could produce them (think: longer outputs mean more chances for an arithmetic
   slip, answer-extraction grabbing the wrong number from a long reasoning chain, or overthinking
   a question that is simpler than it looks).

In [ ]:
# === STUDENT INPUT CELL: Exercise 6.5 ===
# GIVEN starter -- per-question error analysis on the small model (~60 fast API calls).
import pandas as pd

MODEL_SMALL = MODELS[0]   # meta/llama-3.1-8b-instruct

_, zero_preds = evaluate(MODEL_SMALL, zero_shot,         reasoning_dataset)
_, few_preds  = evaluate(MODEL_SMALL, few_shot,          reasoning_dataset, examples=reasoning_fewshot_pool)
_, cot_preds  = evaluate(MODEL_SMALL, chain_of_thought,  reasoning_dataset, examples=reasoning_fewshot_pool)

rows = []
for item, z, f, c in zip(reasoning_dataset, zero_preds, few_preds, cot_preds):
    gold = normalize_numeric(item["answer"])
    rows.append({
        "question": item["question"][:60] + "…",
        "gold": gold,
        "zero": normalize_numeric(z),  "zero_ok": normalize_numeric(z) == gold,
        "few":  normalize_numeric(f),  "few_ok":  normalize_numeric(f) == gold,
        "cot":  normalize_numeric(c),  "cot_ok":  normalize_numeric(c) == gold,
    })
errors_df = pd.DataFrame(rows)

print("Questions where zero-shot AND few-shot are wrong but CoT is right:")
display(errors_df[~errors_df.zero_ok & ~errors_df.few_ok & errors_df.cot_ok])

print("Questions where CoT is wrong but zero-shot OR few-shot is right:")
display(errors_df[~errors_df.cot_ok & (errors_df.zero_ok | errors_df.few_ok)])

# TODO (optional, for part b): repeat for another model on the ladder and compare
# the cot_ok columns across models.

*Your written answers to 6.5(a) and 6.5(b):*

---
## Grading rubric (100 points)

| Exercise | Points | Criteria |
|---|---|---|
| 6.1 — Extend dataset + re-run grid | 20 | 10 valid new items (3+ multi-step, 2+ trick questions); grid re-run without errors; sound written comparison to original results |
| 6.2 — Self-consistency cost/accuracy sweep | 25 | Correct `sweep_self_consistency` reusing `self_consistency`/`chain_of_thought`; both curves plotted; thoughtful discussion of the plateau and its cost implication |
| 6.3 — CoT on a single-step task | 20 | Correct CoT prompt + extractor for sentiment; valid zero-shot vs. CoT comparison on >=2 models; analysis correctly ties task structure to the size of the CoT effect |
| 6.4 — Reflection | 15 | Clearly restates the capability-ladder pattern; identifies a genuine self-consistency failure mode; grounded, non-generic production recommendation |
| Parts 1–5 (harness correctness, run end-to-end) | 20 | Setup runs; dataset, strategies, `self_consistency`, `evaluate`, and the grid all execute without modification and produce a populated `results_df` + plot |
| **Total** | **100** | |

### Submission checklist
- [ ] All setup and given-code cells run top-to-bottom without errors (API key entered, no hardcoded secrets committed)
- [ ] The Part 4 model x strategy grid completed and `results_df` / bar chart are visible in the saved notebook
- [ ] Part 5 analysis answered with reference to your actual numbers
- [ ] Exercises 6.1–6.4 completed, code cells run, written answers filled in
- [ ] Notebook re-run top-to-bottom once before submission (fresh runtime) to confirm no hidden state dependencies

---
### Appendix (optional enrichment)
Try substituting a **different reasoning model** on the gateway (e.g. a DeepSeek-R1-class model) for
`MODELS[-1]` and re-running Part 4. Do the qualitative conclusions from your Part 5 analysis hold, or
does the specific reasoning model matter? This is a good way to test whether your Exercise 6.4-1
answer generalizes beyond the one reasoning model you happened to test.


---
## Part 7 (OPTIONAL) — Base models: pattern completion with your own eyes

Everything above used **instruction-tuned** models — models that were post-trained to follow
instructions. But in the lectures we claimed that underneath, every LLM is a **pattern
completer** ("glorified autocomplete"), and that few-shot learning works *because* it is
pattern completion, not obedience.

NVIDIA's API only serves instruction-tuned models, so to see a genuine **base model** we will
run one *locally*, right here in Colab — no API key needed. We'll use two:

- **GPT-2** (124M parameters, 2019) — from *before instruction tuning existed*
- **Qwen2.5-1.5B** (base variant, 2024) — a modern base model, for a surprising contrast

> ⚙️ For the 1.5B model, switch to a GPU runtime: **Runtime → Change runtime type → T4 GPU**.
> GPT-2 is tiny and runs fine on CPU.

In [ ]:
%pip install -q transformers accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16).to(device)
    return tok, model

def complete(tok, model, prompt, max_new_tokens=60):
    """Greedy continuation of `prompt` -- exactly what a raw language model does."""
    ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

gpt2_tok, gpt2 = load("gpt2")
print("GPT-2 loaded (124M parameters, released 2019)")

### 7.1 — Give a base model an instruction

The prompt below is an *instruction*. An instruction-tuned model answers it. Watch what a
2019 base model does instead.

In [ ]:
print(complete(gpt2_tok, gpt2,
    "Explain the moon landing to a six-year-old in one sentence.", 60))

GPT-2 doesn't *answer* — it **continues the text**, usually rambling or repeating itself.
There is no "assistant" in there; nothing was ever trained to obey. This is exactly the
behavior that motivated instruction tuning (and later RLHF): the InstructGPT paper's famous
example was this same moon-landing prompt.

### 7.2 — Few-shot works anyway

Now give the *same* 2019 base model a few-shot sentiment prompt — the same task you ran
through the API models in Part 3. No instructions, just a pattern:

In [ ]:
sentiment_prompt = (
    "Review: The movie was fantastic! -> positive\n"
    "Review: A total waste of time. -> negative\n"
    "Review: Brilliant acting and a moving story. -> positive\n"
    "Review: The plot made no sense at all. ->"
)
print(complete(gpt2_tok, gpt2, sentiment_prompt, 5))

A 124-million-parameter model from 2019 just did sentiment classification — with no
instruction tuning, no task-specific training, nothing but three examples in the prompt.
**Few-shot learning is pattern completion.** This is precisely the discovery reported in the
GPT-2 and GPT-3 papers, reproduced on your own runtime.

### 7.3 — A *modern* base model blurs the line

Qwen2.5-1.5B (base — not the `-Instruct` variant) was never instruction-tuned either. See how
it handles the same two prompts:

In [ ]:
qwen_tok, qwen = load("Qwen/Qwen2.5-1.5B")   # ~3 GB download, needs the GPU runtime

print("--- instruction ---")
print(complete(qwen_tok, qwen,
    "Explain the moon landing to a six-year-old in one sentence.", 60))

print("\n--- few-shot translation ---")
print(complete(qwen_tok, qwen,
    "Translate English to French.\n\n"
    "sea otter => loutre de mer\n"
    "peppermint => menthe poivrée\n"
    "plush giraffe => girafe en peluche\n"
    "cheese =>", 25))

Two things to notice — both worth discussing:

1. **It answered the instruction**, even though it is a base model. Why? Modern pretraining
   data (2024 web text) is *saturated with instruction-formatted text* — Q&A sites, forum
   answers, even LLM-generated content. The clean 2019 boundary between "base" and
   "instruction-following" has blurred, because the *pattern* of instruction-following is now
   in the pretraining distribution itself.
2. **It doesn't know when to stop.** After the correct answer (`fromage`), it keeps
   completing — inventing more rows, repeating itself. Base models have no concept of an
   end-of-turn; that's part of what chat post-training adds (remember the `<|end|>` markers
   from the Chat Data Structures lecture).

### 7.4 — Reflection (ungraded)

In two or three sentences: your Part 4 grid showed that few-shot prompting barely helped the
instruction-tuned API models, yet here it visibly unlocked a task for GPT-2. Reconcile these
two observations — *when* does few-shot prompting matter, and why?

*Your reflection:*